In [1]:
import os
import numpy as np
import pandas as pd
import networkx as nx
import math
import matplotlib.pyplot as plt
states = ["No motion", "Motion", "Anesthesia"]

In [2]:
from pathlib import Path
from datetime import datetime
FIG_DIR = Path(r"C:/Users/gp00062/OneDrive - WVUM and HSC/WVU postdoc/working on data/processed/figures")
td = datetime.today().strftime("%m_%d_%y")
fig_path = FIG_DIR / td

# 2. Create the directory (this returns None, so we do it on its own line)
fig_path.mkdir(parents=True, exist_ok=True)

In [3]:
FC_signed = {}
net_signed = {}

directory_path = "C:/Users/gp00062/OneDrive - WVUM and HSC/WVU postdoc/codes/figure/deconv"

for state in states:
    safe_state = state.replace(" ", "_")   # "No motion" -> "No_motion"
    matrix_name = f"{directory_path}/adjacency_{safe_state}_deco.npy" # Prepend directory_path
    FC = np.load(matrix_name)
    FC_signed[state] = FC
    print(f"Loaded {matrix_name} with shape {FC.shape}")
    network_name = f"{directory_path}/net_{safe_state}_signed_decon.npy" # Prepend directory_path
    W_signed = np.load(network_name)
    net_signed[state] = W_signed
    print(f"Loaded {network_name} with shape {W_signed.shape}")

Loaded C:/Users/gp00062/OneDrive - WVUM and HSC/WVU postdoc/codes/figure/deconv/adjacency_No_motion_deco.npy with shape (418, 418)
Loaded C:/Users/gp00062/OneDrive - WVUM and HSC/WVU postdoc/codes/figure/deconv/net_No_motion_signed_decon.npy with shape (418, 418)
Loaded C:/Users/gp00062/OneDrive - WVUM and HSC/WVU postdoc/codes/figure/deconv/adjacency_Motion_deco.npy with shape (496, 496)
Loaded C:/Users/gp00062/OneDrive - WVUM and HSC/WVU postdoc/codes/figure/deconv/net_Motion_signed_decon.npy with shape (496, 496)
Loaded C:/Users/gp00062/OneDrive - WVUM and HSC/WVU postdoc/codes/figure/deconv/adjacency_Anesthesia_deco.npy with shape (320, 320)
Loaded C:/Users/gp00062/OneDrive - WVUM and HSC/WVU postdoc/codes/figure/deconv/net_Anesthesia_signed_decon.npy with shape (320, 320)


In [4]:
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica"], # Standard journal fonts
    "font.size": 10,                           # Base font size
    "axes.labelsize": 12,                      # Axis titles
    "axes.titlesize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "axes.linewidth": 1.2,                     # Thickness of the frame
    "lines.linewidth": 1.5,
    "xtick.major.width": 1.2,
    "ytick.major.width": 1.2,
    "pdf.fonttype": 42,                        # Ensures text is editable in Illustrator/Affinity
    "ps.fonttype": 42
})

In [5]:
G_variants = {}
net_types = ["combined", "positive", "negative"]


def build_network_variants(W_signed):
    """
    W_signed: NxN signed, thresholded correlation matrix

    Returns:
        dict of matrices:
            'combined': |W_signed|
            'positive': W_signed with negative edges set to 0
            'negative': -W_signed with positive edges set to 0 (absolute value of negative edges)
    """
    W_combined = np.abs(W_signed)

    W_positive = np.where(W_signed > 0, W_signed, 0.0)

    W_negative = np.where(W_signed < 0, -W_signed, 0.0)  # take absolute value of negatives

    return {
        'combined': W_combined,
        'positive': W_positive,
        'negative': W_negative
    }

def matrix_to_weighted_graph(W):
    """
    W: NxN matrix with non-negative weights.
    Returns a weighted undirected networkx.Graph with 'weight' on edges.
    """
    n = W.shape[0]
    G = nx.Graph()
    G.add_nodes_from(range(n))

    iu = np.triu_indices(n, k=1)
    for i, j in zip(*iu):
        w = W[i, j]
        if w > 0:
            G.add_edge(i, j, weight=float(w))
    return G
    
for state in states:
    print(f"Processing state: {state}")
    G_variants[state] = {}
    W_signed = net_signed[state]
    variants = build_network_variants(W_signed)
    # Convert matrices to NetworkX graph objects and store them
    for net_type in net_types:
        W_variant = variants[net_type]             # <-- use the specific variant matrix
        G = matrix_to_weighted_graph(W_variant)    # convert to weighted graph
        G_variants[state][net_type] = G

        print(f"  Created {net_type} graph for {state} with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.")

print("G_variants dictionary successfully populated.")

Processing state: No motion
  Created combined graph for No motion with 418 nodes and 3697 edges.
  Created positive graph for No motion with 418 nodes and 3455 edges.
  Created negative graph for No motion with 418 nodes and 242 edges.
Processing state: Motion
  Created combined graph for Motion with 496 nodes and 3217 edges.
  Created positive graph for Motion with 496 nodes and 3090 edges.
  Created negative graph for Motion with 496 nodes and 127 edges.
Processing state: Anesthesia
  Created combined graph for Anesthesia with 320 nodes and 3066 edges.
  Created positive graph for Anesthesia with 320 nodes and 2778 edges.
  Created negative graph for Anesthesia with 320 nodes and 288 edges.
G_variants dictionary successfully populated.
